# 13 Search analysis from the shared dataset

**Questions:** Which Singapore search months are comparable, what does Semrush actually observe, and what historical paid-search cautions are available?

Run 04 first. This notebook uses the shared GSC source adjudication, observed query table, Semrush freshness flags and historical Google Ads labels. It does not choose workbook copies again, impute absent queries as zero, merge feature visibility with traditional organic rankings, or create an automatic do-not-bid list.

Source selection and coverage are part of each result. Historical rankings and red-list flags warrant investigation; they do not prove current paid-search cannibalization.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # run from the notebooks folder

import pandas as pd
from IPython.display import display
from weavve.clean import load_dataset

dataset = load_dataset()
print("Shared dataset ID:", dataset.manifest["dataset_id"])
print("Local dataset:", dataset.path)
print("Transaction cutoff:", dataset.manifest["transaction_cutoff"])
print("Review flags:", len(dataset.manifest["warnings"]))
pd.set_option("display.max_columns", 20)


## Search Console source selection and query coverage

The monthly table contains only periods accepted by the shared Singapore/Web/full-month policy. The candidate table retains exclusions and superseded sources. Observed query sums and coverage use only rows with a recorded click value; their missing-row counts remain visible. The complete exported-query sum and coverage are unknown if any query row is missing a value. Neither measure represents every search query: the source export itself is incomplete. Missing months, queries and cells are never converted to zero here.


In [ ]:
gsc = dataset.frame("gsc_months")
candidates = dataset.frame("gsc_month_candidates")
display(candidates.groupby(["selected", "reason"], dropna=False).size().rename("candidate_rows").to_frame())
monthly = gsc[[
    "month", "clicks", "impressions", "query_rows", "observed_query_rows", "missing_query_rows",
    "observed_query_clicks_sum", "observed_query_click_coverage", "query_clicks_sum", "query_click_coverage", "reason",
]].sort_values("month")
display(monthly)
print("Selected calendar months:", len(monthly))


## Semrush visibility and observation dates

The filename month and row timestamp are separate. Count keywords distinctly within each month and visibility/freshness group; the same keyword may appear in several groups, so the rows of this summary are not additive. "Fresh" here only means the timestamp is not earlier than the filename month, not a guarantee of a new measurement. Unknown and after-cutoff timestamps remain visible.


In [ ]:
semrush = dataset.frame("semrush_keywords")
sem_summary = semrush.groupby(["file_month", "visibility_class", "is_stale_observation", "row_after_cutoff"], dropna=False).agg(
    source_rows=("source_row", "size"), distinct_keywords=("Keyword", "nunique"),
)
display(sem_summary)

observed_top10 = semrush.loc[semrush["clean_position"].between(1, 10)]
top10_summary = observed_top10.groupby(["file_month", "visibility_class", "is_stale_observation"], dropna=False)["Keyword"].nunique().rename("distinct_top10_keywords")
display(top10_summary.to_frame())


## Historical Google Ads context

File 12 has an unspecified all-time window. Its impression-weighted CTR is not a matched benchmark for a later flight. Missing metric cells are counted per group; all-missing totals remain unknown and CTR is suppressed if either metric has any missing component. The red-list field preserves the supplier's historical caution and the negative-keyword table preserves existing exclusions. This summary neither recommends spending nor changes any ad account.


In [ ]:
ads = dataset.frame("ads_keywords")
paid_summary = ads.groupby("historical_red_list_flag", dropna=False).agg(
    keyword_adgroup_rows=("source_row", "size"), distinct_keywords=("clean_keyword", "nunique"),
    impressions=("clean_impressions", lambda s: s.sum(min_count=1)),
    clicks=("clean_clicks", lambda s: s.sum(min_count=1)),
    missing_impressions=("clean_impressions", lambda s: int(s.isna().sum())),
    missing_clicks=("clean_clicks", lambda s: int(s.isna().sum())),
)
paid_summary["weighted_ctr_pct"] = [
    100 * r.clicks / r.impressions
    if r.missing_impressions == 0 and r.missing_clicks == 0 and r.impressions > 0 else None
    for r in paid_summary.itertuples()
]
display(paid_summary.round(2))
negatives = dataset.frame("ads_negatives")
display(pd.DataFrame([{
    "negative_keyword_rows": len(negatives),
    "distinct_normalized_negative_terms": negatives["clean_keyword"].nunique(),
    "ads_rows_with_missing_impressions": int(ads["clean_impressions"].isna().sum()),
    "ads_rows_with_missing_clicks": int(ads["clean_clicks"].isna().sum()),
}]))
